# Diffusion Models

This notebook accompanies the **ML Viz** lesson on diffusion models.
We'll implement a simple diffusion process and learn to reverse it.

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/generative-models/05-diffusion-models

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Intuition — generation by learning to undo noise

Diffusion models generate by reversing destruction. The **forward process** gradually adds Gaussian
noise over `T` steps until the data is indistinguishable from pure noise — no learning involved, and a
closed form jumps to any step: `x_t = √ᾱ_t·x₀ + √(1−ᾱ_t)·ε`. The **reverse process** is where the
model lives: a network is trained on the dead-simple objective *"predict the noise `ε` that was
added"* (an MSE regression!), and sampling walks backward from pure noise, subtracting a little
predicted noise at each step (plus a calibrated random kick). Stable to train (no adversarial game),
likelihood-grounded, and the engine of DALL·E/Stable Diffusion/Imagen. We build the whole pipeline
and validate the learned denoiser against its closed-form optimum.

## The Forward Process (Adding Noise)

The forward process gradually adds Gaussian noise to data:

$$x_t = \sqrt{1 - \beta_t} \cdot x_{t-1} + \sqrt{\beta_t} \cdot \epsilon_t$$

After $T$ steps, the data is pure noise. Let's visualize this on 2D data.

In [ ]:
np.random.seed(42)

# Create 2D data: a circle
n = 200
theta = np.linspace(0, 2 * np.pi, n)
X0 = np.column_stack([2 * np.cos(theta), 2 * np.sin(theta)]) + 0.1 * np.random.randn(n, 2)

# Define noise schedule
T = 100
betas = np.linspace(0.0001, 0.06, T)   # schedule chosen so alpha_bar_T ~ 0.05 (near pure noise)
alphas = 1 - betas
alpha_bars = np.cumprod(alphas)

def add_noise(x0, t):
    """Add noise to x0 at timestep t."""
    ab = alpha_bars[t]
    noise = np.random.randn(*x0.shape)
    return np.sqrt(ab) * x0 + np.sqrt(1 - ab) * noise, noise

# Show snapshots
timesteps = [0, 10, 30, 60, 99]
fig, axes = plt.subplots(1, len(timesteps), figsize=(4 * len(timesteps), 4))
fig.suptitle('Forward Process: Adding Noise', color='white', fontsize=13, y=1.02)

for ax, t in zip(axes, timesteps):
    if t == 0:
        xt = X0
    else:
        xt, _ = add_noise(X0, t)
    ax.scatter(xt[:, 0], xt[:, 1], c='#818cf8', s=10, alpha=0.6)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect('equal')
    ab = alpha_bars[t]
    ax.set_title(f't={t}  ($\\bar{{\\alpha}}$={ab:.3f})', color='white', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

**What to notice:** the ring dissolves smoothly into an isotropic Gaussian blob as `ᾱ_t` falls
toward 0 — by the last step essentially no signal remains, which is essential: **sampling starts from
pure noise**, so the forward process must actually end there. (The schedule is tuned so `ᾱ_T ≈ 0.05`;
a too-short schedule that stops at `ᾱ_T ≈ 0.6` leaves a gap the reverse process can never bridge.)

## Noise schedule

The $\beta_t$ schedule controls how quickly noise is added.
$\bar{\alpha}_t = \prod_{s=1}^t (1 - \beta_s)$ is the cumulative signal remaining.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(betas, color='#f43f5e', linewidth=2)
axes[0].set_title('$\\beta_t$ (noise rate per step)', color='white', fontsize=12)
axes[0].set_xlabel('Timestep $t$')

axes[1].plot(alpha_bars, color='#818cf8', linewidth=2)
axes[1].set_title('$\\bar{\\alpha}_t$ (signal remaining)', color='white', fontsize=12)
axes[1].set_xlabel('Timestep $t$')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

print(f'At t=0:  signal = {alpha_bars[0]:.4f} (almost clean)')
print(f'At t={T-1}: signal = {alpha_bars[-1]:.6f} (almost pure noise)')

**What to notice:** `β_t` ramps up linearly while `ᾱ_t = Π(1−β)` decays toward 0 — small careful
steps early (where structure is being erased) and bigger ones later. The `ᾱ` curve is the one that
matters: it is exactly the fraction of original signal surviving at step `t`.

## The Reverse Process (Denoising)

The reverse process learns to remove noise step by step.
For our simple 2D data, we can learn the denoising function directly.

The key equation for each reverse step:

$$x_{t-1} = \frac{1}{\sqrt{1 - \beta_t}} \left(x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}} \epsilon_\theta(x_t, t)\right) + \sigma_t z$$

In [ ]:
class SimpleDenoiser:
    """Per-timestep linear noise predictors: eps_hat = x_t @ W[t] + b[t]."""

    def __init__(self):
        self.W = {}
        self.b = {}

    def predict_noise(self, x, t):
        if t in self.W:
            return x @ self.W[t] + self.b[t]
        return np.zeros_like(x)

    def train(self, X0, T, betas, alphas, alpha_bars, epochs=500, lr=0.02):
        """Train each timestep's denoiser by SGD on 'predict the added noise' (MSE).
        Weights PERSIST across epochs — each epoch continues from the last."""
        losses = []
        for epoch in range(epochs):
            epoch_loss = 0.0
            for t in range(1, T):
                ab = alpha_bars[t]
                noise = np.random.randn(*X0.shape)
                Xt = np.sqrt(ab) * X0 + np.sqrt(1 - ab) * noise

                W = self.W.get(t, np.zeros((2, 2)))     # continue from current weights
                b = self.b.get(t, np.zeros(2))

                pred = Xt @ W + b
                err = pred - noise
                epoch_loss += np.mean(err ** 2)

                W = W - lr * (Xt.T @ err / len(X0))
                b = b - lr * err.mean(axis=0)
                self.W[t] = W
                self.b[t] = b
            losses.append(epoch_loss / (T - 1))
        return losses

denoiser = SimpleDenoiser()
losses = denoiser.train(X0, T, betas, alphas, alpha_bars, epochs=500, lr=0.02)
print(f'denoising MSE: epoch 1 = {losses[0]:.3f}  ->  epoch 500 = {losses[-1]:.3f}  (noise variance baseline = 1.0)')

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(losses, color='#818cf8', linewidth=1.5)
ax.set_title('Denoiser Training Loss', color='white', fontsize=12)
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE vs true noise')
plt.tight_layout(); plt.show()

**What to notice:** the objective really is just **MSE against the noise you injected** — no
adversary, no ELBO gymnastics — and the loss drops well below the variance-1 baseline as each
timestep's denoiser learns. (This cell also fixes a bug in the original notebook, which re-initialized
the weights to zero every epoch, so nothing ever accumulated.)

## The library way — validate against the closed-form optimal denoiser

For this toy the *optimal linear* noise predictor has a closed form: with data covariance `σ₀²I`
(here `σ₀² = 2` for the ring), the best `W[t]` is `√(1−ᾱ_t) / (ᾱ_t·σ₀² + (1−ᾱ_t)) · I`. The check:
our SGD-trained weights must converge to it.

In [ ]:
sigma0_sq = X0.var(axis=0).mean()          # ~2 for the radius-2 ring
print(f'data variance per dim ~ {sigma0_sq:.2f}')
for t in [20, 50, 80]:
    ab = alpha_bars[t]
    w_opt = np.sqrt(1 - ab) / (ab * sigma0_sq + (1 - ab))
    w_learned = np.diag(denoiser.W[t]).mean()
    print(f't={t:>3}: learned W diag = {w_learned:.3f}   closed-form optimum = {w_opt:.3f}')
    assert abs(w_learned - w_opt) < 0.05, "SGD must converge to the optimal linear denoiser"
print('\nthe trained denoiser matches the closed-form optimal linear predictor ✓')

**What to notice:** at every checked timestep the learned weight sits on the closed-form optimum —
the simple "predict the noise" regression is solving exactly the problem the theory says it solves.
For images, the same objective is optimized by a U-Net instead of a 2×2 matrix; nothing else changes.

## Sampling: Reverse the diffusion

Start from pure noise and iteratively denoise.

In [ ]:
def sample(denoiser, n_samples, T, betas, alpha_bars):
    """Generate samples by reversing the diffusion process."""
    x = np.random.randn(n_samples, 2)  # start from pure noise
    trajectory = [x.copy()]
    
    for t in range(T - 1, 0, -1):
        # Predict noise
        eps_pred = denoiser.predict_noise(x, t)
        
        # Reverse step
        beta_t = betas[t]
        ab = alpha_bars[t]
        sigma = np.sqrt(beta_t)
        
        x = (1 / np.sqrt(1 - beta_t)) * (x - (beta_t / np.sqrt(1 - ab)) * eps_pred)
        if t > 1:
            x = x + sigma * np.random.randn(*x.shape)   # DDPM stochastic term (omit at the last step)
        
        if t % 20 == 0:
            trajectory.append(x.copy())
    
    return x, trajectory

x_gen, trajectory = sample(denoiser, 200, T, betas, alpha_bars)

# Show the reverse process
fig, axes = plt.subplots(1, len(trajectory), figsize=(4 * len(trajectory), 4))
fig.suptitle('Reverse Process: Denoising to Generation', color='white', fontsize=13, y=1.02)

for ax, x_step in zip(axes, trajectory):
    ax.scatter(x_step[:, 0], x_step[:, 1], c='#14b8a6', s=10, alpha=0.6)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect('equal')
    ax.axis('off')

axes[0].set_title('Pure Noise', color='white', fontsize=10)
axes[-1].set_title('Generated', color='white', fontsize=10)

plt.tight_layout()
plt.show()

**What to notice:** generation runs the chain **backward** — subtract a fraction of the predicted
noise, then (crucially) add back a small calibrated random kick `σ_t·z`. Without that stochastic term
the reverse process over-contracts and the samples end up too tight; with it, the trajectory panels
show noise condensing step by step onto the data distribution. (The original sampler omitted this
term — another quiet bug fixed here.)

## Real vs Generated

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(X0[:, 0], X0[:, 1], c='#818cf8', s=15, alpha=0.6)
axes[0].set_title('Real Data', color='white', fontsize=12)
axes[0].set_xlim(-4, 4)
axes[0].set_ylim(-4, 4)
axes[0].set_aspect('equal')

axes[1].scatter(x_gen[:, 0], x_gen[:, 1], c='#14b8a6', s=15, alpha=0.6)
axes[1].set_title('Generated (Diffusion)', color='white', fontsize=12)
axes[1].set_xlim(-4, 4)
axes[1].set_ylim(-4, 4)
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

**What to notice:** the generated cloud matches the ring's **scale and spread** (radius ≈ 2), though
a *linear* per-step denoiser can only shape Gaussian statistics — it fills the ring's disk rather than
carving its crisp circle. That residual gap is precisely what the U-Net buys in a real diffusion
model: a non-linear `ε`-predictor that can bend the reverse flow onto thin manifolds.

## Gotchas & tradeoffs

- **The schedule must reach noise.** If `ᾱ_T` isn't ≈ 0, sampling (which starts at pure noise) begins
  in a state the model never trained on — a silent quality killer (the bug this notebook originally
  had).
- **The reverse noise term is not optional.** Deterministic reverse steps with a DDPM-trained model
  over-contract; DDIM makes determinism *correct* but changes the update equations.
- **Sampling is slow** — one network pass per step (hundreds to thousands). Fast samplers (DDIM,
  distillation, consistency models) exist precisely for this.
- **Stable but compute-hungry:** vs GANs, diffusion trades adversarial instability for many-step
  sampling cost; vs VAEs, it wins sharpness at the same likelihood-based safety.

In [ ]:
# The schedule gotcha, quantified: how much SIGNAL survives at t=T for different schedules
for T_s, bmax in [(50, 0.02), (50, 0.08), (100, 0.06)]:
    ab_T = np.cumprod(1 - np.linspace(1e-4, bmax, T_s))[-1]
    print(f'T={T_s:>3}, beta_max={bmax}: alpha_bar_T = {ab_T:.3f}  -> {np.sqrt(ab_T)*100:4.0f}% signal left at "pure noise"')
print('\n-> the original T=50/0.02 schedule left ~78% signal scale at t=T: sampling from N(0,1) then')
print('   starts far off-distribution. Always check alpha_bar_T ~ 0 before trusting a diffusion toy.')

**What to notice:** the original `T=50, β≤0.02` schedule leaves **78% of the signal scale** at the
final step — "pure noise" that isn't. The corrected schedule leaves ~5%. Checking `ᾱ_T` is a one-line
sanity test every diffusion implementation should run; real systems use cosine schedules designed for
exactly this property.

## Key takeaways

1. **Forward process**: gradually add noise until data becomes pure Gaussian
2. **Reverse process**: learn to denoise step by step
3. **Training**: predict the noise $\epsilon_\theta(x_t, t)$ — simple MSE loss
4. **Sampling**: iterative — slower than GANs but higher quality and diversity
5. **Connection to score matching**: the denoiser learns $\nabla_x \log P(x)$

This is the foundation behind DALL·E, Stable Diffusion, and Midjourney.

## ✏️ Your turn

### Exercise 1 — Forward diffusion noise level

The closed-form forward process jumps directly to timestep $t$:

$$x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\, \varepsilon, \quad \varepsilon \sim \mathcal{N}(0,I)$$

where $\bar\alpha_t = \prod_{s=1}^{t}(1-\beta_s)$.  
As $t \to T$, $\bar\alpha_t \to 0$ and the signal is overwhelmed by noise.
Verify the boundary conditions using the notebook's beta schedule.

In [ ]:
import numpy as np

T_steps = 100
betas = np.linspace(0.0001, 0.06, T_steps)
alpha_bars = np.cumprod(1 - betas)   # reuse the notebook's schedule

def add_noise(x0, t, seed=0):
    """Add noise to x0 at timestep t (0-indexed) using the closed-form formula.
    x0: array, t: int, returns noisy array same shape as x0."""
    # TODO(you): sample eps ~ N(0,1) then apply the closed-form one-step jump
    ...

In [ ]:
# Boundary conditions on alpha_bars
assert alpha_bars[0] > 0.99, \
    "alpha_bar at t=0 must be ~1 (almost no noise yet)"
assert alpha_bars[-1] < 0.05, \
    "alpha_bar at t=T-1 must be ~0 (mostly noise)"

x0 = np.array([[2.0, 0.0]])
x_t0 = add_noise(x0, t=0, seed=0)
assert np.linalg.norm(x_t0 - x0) < 0.5, \
    "at t=0 the noisy sample should be close to x0 (small beta)"
# late-step distance is random per seed, so check the AVERAGE over many seeds
dists_T = [np.linalg.norm(add_noise(x0, t=99, seed=s) - x0) for s in range(50)]
assert np.mean(dists_T) > 1.0, \
    "at t=T-1 the noisy sample is far from x0 on average (signal destroyed)"
assert add_noise(x0, t=0, seed=0).shape == x0.shape, \
    "output shape must match x0"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def add_noise(x0, t, seed=0):
    rng = np.random.RandomState(seed)
    eps = rng.randn(*x0.shape)
    return np.sqrt(alpha_bars[t]) * x0 + np.sqrt(1 - alpha_bars[t]) * eps
```

</details>

### Exercise 2 — Reverse (denoising) step

Given the predicted noise $\hat\varepsilon = \varepsilon_\theta(x_t, t)$, the reverse step is:

$$x_{t-1} = \frac{1}{\sqrt{1-\beta_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\hat\varepsilon\right) + \sigma_t z,\quad z \sim \mathcal{N}(0,I)$$

where $\sigma_t = \sqrt{\beta_t}$. Implement this and verify the deterministic (z=0) case.

In [ ]:
def reverse_step(x_t, t, eps_hat, seed=0, stochastic=True):
    """One reverse diffusion step from x_t → x_{t-1}.
    t: current timestep (1-indexed here, so t=1 is the final step).
    eps_hat: predicted noise array same shape as x_t.
    If stochastic=False, sets z=0 (deterministic DDIM-like)."""
    # TODO(you): implement the reverse step formula
    # beta_t = betas[t-1], alpha_bar_t = alpha_bars[t-1]
    ...

In [ ]:
x_T = np.array([[0.5, -0.3]])
t_step = 25
# Use a zero noise predictor (eps_hat = 0) for easy hand-checking
x_prev_det = reverse_step(x_T, t_step, eps_hat=np.zeros_like(x_T), stochastic=False)

assert x_prev_det.shape == x_T.shape, "output shape must equal input shape"

# With eps_hat=0, the formula reduces to x_T / sqrt(1 - beta_t)
beta_t = betas[t_step - 1]
expected = x_T / np.sqrt(1 - beta_t)
assert np.allclose(x_prev_det, expected, atol=1e-9), \
    "with zero predicted noise, denoising step = x_t / sqrt(1-beta_t)"

# Stochastic step has different output from deterministic (due to added noise)
np.random.seed(1)
x_prev_sto = reverse_step(x_T, t_step, eps_hat=np.zeros_like(x_T), stochastic=True)
assert not np.allclose(x_prev_det, x_prev_sto, atol=1e-6), \
    "stochastic and deterministic steps should differ when sigma_t > 0"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def reverse_step(x_t, t, eps_hat, seed=0, stochastic=True):
    beta_t      = betas[t - 1]
    alpha_bar_t = alpha_bars[t - 1]
    coeff       = beta_t / np.sqrt(1 - alpha_bar_t)
    x_prev      = (x_t - coeff * eps_hat) / np.sqrt(1 - beta_t)
    if stochastic:
        rng = np.random.RandomState(seed)
        x_prev = x_prev + np.sqrt(beta_t) * rng.randn(*x_t.shape)
    return x_prev
```

</details>